This notebook was run last on the following commit

In [ ]:
!git log -1

In [ ]:
%matplotlib inline

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotnine as gg
import numpy as np
import sys

sys.path.append("/workspace")


from src.evaluation.kernel_evaluation import (
    compute_distance_matrix,
    process_and_align,
    wide_to_long,
)

In [ ]:
# metabolic_kernel_path = "/workspace/results/ecoli_rich_medium/fba_moma/default/kernel.pkl"
metabolic_kernel_path = "/workspace/results/ecoli_rich_medium/gene_graph/beta_1/kernel.pkl"
reference_kernel_path = "/workspace/results/ecoli_rich_medium/targets/transcriptomic_kernel.pkl"

metabolic_kernel = pd.read_pickle(metabolic_kernel_path)
metabolic_dist = compute_distance_matrix(metabolic_kernel)
reference_kernel = pd.read_pickle(reference_kernel_path)
reference_dist = compute_distance_matrix(reference_kernel)

metabolic_kernel_, reference_kernel_ = process_and_align(metabolic_kernel, reference_kernel)
metabolic_dist_, reference_dist_ = process_and_align(metabolic_dist, reference_dist)

In [ ]:
metabolic_dist_long = wide_to_long(
    metabolic_dist_,
    "distance_pred",
    "gene1",
    "gene2",
    remove_diagonal=True,
    remove_lower_triangle=True,
)

reference_dist_long = wide_to_long(
    reference_dist_,
    "distance_target",
    "gene1",
    "gene2",
    remove_diagonal=True,
    remove_lower_triangle=True,
)
joint_long = pd.merge(metabolic_dist_long, reference_dist_long, on=["gene1", "gene2"], how="inner")

In [ ]:
plt.scatter(joint_long["distance_pred"], joint_long["distance_target"], alpha=0.5)

In [ ]:
import scipy.stats as stats

stats.spearmanr(joint_long["distance_pred"], joint_long["distance_target"])

In [ ]:
metabolic_dist_long = metabolic_dist_long.sample(frac=1.0, replace=False)
for _, row in metabolic_dist_long.sort_values("distance_pred").head(50).iterrows():
    print(f"{row['gene1']} - {row['gene2']}")

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, metric="precomputed", init="random")
metabolic_kernel_tsne = tsne.fit_transform(metabolic_kernel_)

In [ ]:
plot_df = pd.DataFrame(
    metabolic_kernel_tsne, index=metabolic_kernel_.index, columns=["TSNE1", "TSNE2"]
).assign(gene_name=lambda x: x.index)

(gg.ggplot(plot_df, gg.aes(x="TSNE1", y="TSNE2")) + gg.geom_point() + gg.theme_minimal())

In [ ]:
import plotly.express as px


fig = px.scatter(
    plot_df,
    x="TSNE1",
    y="TSNE2",
    hover_name="gene_name",
    template="plotly_white",
    width=600,
    height=400,
)
fig.update_traces(marker=dict(size=3))
fig.show()

In [ ]:
gene1 = "menH"
gene2 = "sseA"
selected_dist = metabolic_dist_.loc[gene1, gene2].item()

triu_indices = np.triu_indices(metabolic_dist_.shape[0], k=1)
metabolic_dist_triu = metabolic_dist_.values[triu_indices]
plot_df = pd.DataFrame(
    {
        "dist": metabolic_dist_triu,
    }
)

(
    gg.ggplot(plot_df, gg.aes(x="dist"))
    + gg.geom_histogram(bins=100)
    + gg.theme_minimal()
    + gg.scale_y_log10()
    + gg.geom_vline(xintercept=selected_dist, color="red")
)
# plt.hist(metabolic_dist_triu, bins=100)
# plt.yscale("log")
# plt.show()

In [ ]:
metabolic_dist_.loc[gene1, gene2].item()

In [ ]:
gene1 = "leuC"
gene2 = "leuD"
reference_dist_.loc[gene1, gene2] / reference_dist_.mean().mean()

In [ ]:
gene1 = "lpxA"
gene2 = "lpxK"
reference_dist_.loc[gene1, gene2] / reference_dist_.mean().mean()

In [ ]:
res = metabolic_dist_long.sort_values("dist").head(50)

for _, row in res.iterrows():
    print(f"{row['gene1']} - {row['gene2']}")

In [ ]:
gene_pairs = res["gene_pair"]
reference_subset = reference_dist_long.loc[reference_dist_long["gene_pair"].isin(gene_pairs)]

(
    gg.ggplot(reference_dist_long, gg.aes(x="dist", y=gg.after_stat("density")))
    + gg.geom_histogram(bins=100)
    + gg.geom_histogram(reference_subset, fill="red", alpha=0.5, bins=50)
    + gg.theme_minimal()
)

In [ ]:
import scanpy as sc

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata_case = sc.read_h5ad("/workspace/data/251117_genomescale_CRISPRi/adata_case.annotated.h5ad")

In [ ]:
adata.obs["umap_x"] = adata.obsm["X_umap"][:, 0]
adata.obs["umap_y"] = adata.obsm["X_umap"][:, 1]

In [ ]:
adata.X = adata.layers["cp10k"]
sc.tl.pca(adata)

adata.obs["pc1"] = adata.obsm["X_pca"][:, 0]
adata.obs["pc2"] = adata.obsm["X_pca"][:, 1]


gene1 = "leuC"
gene2 = "leuD"

adata_sub = adata[adata.obs["gene"].isin([gene1, gene2])]
fig = (
    gg.ggplot(
        adata.obs,
        gg.aes(x="pc1", y="pc2"),
    )
    + gg.geom_point()
    + gg.geom_point(adata_sub.obs, gg.aes(x="pc1", y="pc2", color="gene"))
)
display(fig)

In [ ]:
adata.X = adata.layers["cp10k"]
sc.tl.pca(adata)

adata.obs["pc1"] = adata.obsm["X_pca"][:, 0]
adata.obs["pc2"] = adata.obsm["X_pca"][:, 1]


gene1 = "nusA"
gene2 = "nusG"

adata_sub = adata[adata.obs["gene"].isin([gene1, gene2])]
fig = (
    gg.ggplot(
        adata.obs,
        gg.aes(x="pc1", y="pc2"),
    )
    + gg.geom_point()
    + gg.geom_point(adata_sub.obs, gg.aes(x="pc1", y="pc2", color="gene"))
)
display(fig)

In [ ]:
display(reference_subset.sort_values("dist"))

for _, row in reference_subset.sort_values("dist").iterrows():
    print(f"{row['gene1']} - {row['gene2']}")

In [ ]:
gene1 = "leuC"
gene2 = "leuD"

adata_sub = adata[adata.obs["gene"].isin([gene1, gene2])]
fig = (
    gg.ggplot(
        adata.obs,
        gg.aes(x="umap_x", y="umap_y"),
    )
    + gg.geom_point()
    + gg.geom_point(adata_sub.obs, gg.aes(x="umap_x", y="umap_y", color="gene"))
)
display(fig)

In [ ]:
adata_case

In [ ]:
gene1 = "hemB"
gene2 = "hemL"

adata_sub = adata_case[adata_case.obs["gene"].isin([gene1, gene2])]
fig = (
    gg.ggplot(
        adata_case.obs,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"),
    )
    + gg.geom_point()
    + gg.geom_point(
        adata_sub.obs, gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="gene")
    )
)
display(fig)

In [ ]:
gene1 = "hemB"
gene2 = "hemL"

adata_sub = adata[adata.obs["gene"].isin([gene1, gene2])].copy()
sc.tl.rank_genes_groups(adata_sub, "gene", method="wilcoxon")
de_results = sc.get.rank_genes_groups_df(adata_sub, group=gene1)
de_results.sort_values("pvals").head(50).loc[:, ["names", "pvals", "logfoldchanges", "pvals_adj"]]

In [ ]:
adata_sub

In [ ]:
gene_name = "rpsC"
adata_sub.obs.loc[:, gene_name] = adata_sub[:, gene_name].layers["cp10k"].toarray().flatten()
(
    gg.ggplot(
        adata_sub.obs,
        gg.aes(x=gene_name, fill="gene", y=gg.after_stat("density")),
    )
    + gg.geom_histogram(bins=5, position="identity", alpha=0.5)
    + gg.theme_minimal()
)

In [ ]:
triu_indices = np.triu_indices(metabolic_dist_.shape[0], k=1)
metabolic_dist_triu = metabolic_dist_.values[triu_indices]
plt.hist(metabolic_dist_triu, bins=100)
plt.yscale("log")
plt.show()

In [ ]:
gene1 = "nusA"
gene2 = "nusG"

adata_sub = adata[adata.obs["gene"].isin([gene1, gene2])]
fig = (
    gg.ggplot(
        adata.obs,
        gg.aes(x="umap_x", y="umap_y"),
    )
    + gg.geom_point()
    + gg.geom_point(adata_sub.obs, gg.aes(x="umap_x", y="umap_y", color="gene"))
)
display(fig)